In [2]:
!wget -q -O hetionet-nodes.tsv https://github.com/hetio/hetionet/raw/master/hetnet/tsv/hetionet-v1.0-nodes.tsv
!wget -q -O hetionet-edges.sif.gz https://github.com/hetio/hetionet/raw/master/hetnet/tsv/hetionet-v1.0-edges.sif.gz

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import random

nodes_df = pd.read_csv('hetionet-nodes.tsv', sep='\t')
edges_df = pd.read_csv('hetionet-edges.sif.gz', sep='\t', compression='gzip')

# dropping nulls and exact duplicates to ensure strict structural integrity
nodes_df.dropna(inplace=True)
edges_df.dropna(inplace=True)
nodes_df.drop_duplicates(inplace=True)
edges_df.drop_duplicates(inplace=True)

# preserving all biological entities to maximize the effectiveness of downstream centrality and community detection algorithms
valid_node_ids = set(nodes_df['id'])
filtered_edges = edges_df[
    edges_df['source'].isin(valid_node_ids) &
    edges_df['target'].isin(valid_node_ids)
].copy()

# calculating baseline topological features across the entire heterogeneous network
out_degree = filtered_edges['source'].value_counts().rename('out_degree')
in_degree = filtered_edges['target'].value_counts().rename('in_degree')

filtered_nodes = nodes_df.merge(out_degree, how='left', left_on='id', right_index=True)
filtered_nodes = filtered_nodes.merge(in_degree, how='left', left_on='id', right_index=True)
filtered_nodes['out_degree'] = filtered_nodes['out_degree'].fillna(0).astype(int)
filtered_nodes['in_degree'] = filtered_nodes['in_degree'].fillna(0).astype(int)

# isolating compound and disease pairs to formulate the specific link prediction target
positive_edges = filtered_edges[filtered_edges['metaedge'] == 'CtD'].copy()
positive_edges = positive_edges[['source', 'target']]
positive_edges['label'] = 1

compounds = filtered_nodes[filtered_nodes['kind'] == 'Compound']['id'].tolist()
diseases = filtered_nodes[filtered_nodes['kind'] == 'Disease']['id'].tolist()
positive_pairs_set = set(zip(positive_edges['source'], positive_edges['target']))

negative_pairs = []
num_positive = len(positive_edges)

# generating balanced negative samples by pairing compounds and diseases with no known therapeutic link
while len(negative_pairs) < num_positive:
    c = random.choice(compounds)
    d = random.choice(diseases)

    if (c, d) not in positive_pairs_set:
        negative_pairs.append((c, d))
        positive_pairs_set.add((c, d))

negative_edges = pd.DataFrame(negative_pairs, columns=['source', 'target'])
negative_edges['label'] = 0

ml_dataset = pd.concat([positive_edges, negative_edges], ignore_index=True)

# merging foundational node metrics into the training matrix
ml_dataset = ml_dataset.merge(
    filtered_nodes[['id', 'out_degree', 'in_degree']],
    left_on='source', right_on='id', how='left'
).rename(columns={'out_degree': 'compound_out_degree', 'in_degree': 'compound_in_degree'})

ml_dataset.drop(columns=['id'], inplace=True)

ml_dataset = ml_dataset.merge(
    filtered_nodes[['id', 'out_degree', 'in_degree']],
    left_on='target', right_on='id', how='left'
).rename(columns={'out_degree': 'disease_out_degree', 'in_degree': 'disease_in_degree'})

ml_dataset.drop(columns=['id'], inplace=True)

# exporting the complete heterogeneous network and the targeted machine learning matrix for subsequent phases
filtered_nodes.to_csv('neo4j_nodes.csv', index=False)
filtered_edges.to_csv('neo4j_edges.csv', index=False)
ml_dataset.to_csv('baseline_ml_dataset.csv', index=False)

print("\nExecution Complete.")
print(f"Total ML Training Rows: {len(ml_dataset)}")
print("Files 'neo4j_nodes.csv', 'neo4j_edges.csv', and 'baseline_ml_dataset.csv' are ready.")


Execution Complete.
Total ML Training Rows: 1510
Files 'neo4j_nodes.csv', 'neo4j_edges.csv', and 'baseline_ml_dataset.csv' are ready.
